# BTS Flight Data Ingestion - Bronze Layer

This notebook implements the **Bronze layer** of the medallion architecture for US flight data from the Bureau of Transportation Statistics (BTS).

## Medallion Architecture Overview

```
RAW DATA → BRONZE (this notebook) → SILVER → GOLD
```

* **Bronze**: Raw ingestion with minimal transformation (schema enforcement, partitioning)
* **Silver**: Data quality rules, cleansing, standardization
* **Gold**: Business-level aggregations and metrics

## This Notebook's Role

**Layer**: Bronze (Raw Data Ingestion)  
**Source**: BTS TranStats On-Time Performance API  
**Output Table**: `skyops.bronze.bts_ontime`  
**Output Type**: Managed Delta Table (Unity Catalog)  
**Transformation**: Minimal - schema enforcement, date normalization, partitioning  
**Time Range**: Jan 2019 to latest available month (dynamic)  
**Expected Runtime**: 3-4 hours (initial full download) + 10-15 minutes (Delta table creation)

## Architecture Flow

```
BTS TranStats API
    ↓
[bts_downloader.py] ← Python script (ingestion/)
    ↓
/Volumes/skyops/bronze/bts/staging/*.csv ← Staging layer (monthly CSVs)
    ↓
[THIS NOTEBOOK] ← Bronze transformation
    ↓
skyops.bronze.bts_ontime ← Bronze Delta table
    ↓
[Future: silver/bts_ontime_clean.py] ← Silver layer
```

## Catalog Structure

```
skyops (catalog)
├── bronze (schema)              # ← Bronze layer output
│   ├── bts_ontime              # This notebook creates this
│   └── bts (volume)            # Staging CSVs stored here
├── silver (schema)              # Future: cleaned data
└── gold (schema)                # Future: business metrics
```

## Execution Steps

1. **Install dependencies** (Cell 2)
2. **Download staging CSVs** (Cell 5) - Automatically downloads from Jan 2019 to latest available
3. **Create Bronze Delta table** (Cell 7) - Combines CSVs with schema enforcement
4. **Validate data integrity** (Cells 10-12) - Schema, row count, and data quality validation

## Prerequisites

* Unity Catalog Volume: `skyops.bronze.bts` (must exist)
* Databricks Serverless or cluster with internet access
* 3-4 hours runtime for initial full download

## Bronze Layer Characteristics

✓ **Raw data preservation**: Minimal transformation  
✓ **Schema enforcement**: Explicit types prevent drift  
✓ **Date normalization**: Standardized yyyyMMdd format  
✓ **Partitioning**: YEAR-based for query performance  
✓ **Idempotency**: Re-running overwrites safely  
✓ **Resume capability**: Already-downloaded months are skipped automatically  
✓ **Dynamic date range**: Automatically fetches from Jan 2019 to latest available month

In [0]:
%pip install requests beautifulsoup4 lxml python-dateutil pyarrow --quiet
dbutils.library.restartPython()

In [0]:
%sql
USE skyops.bronze

In [0]:
import subprocess
import sys
from pyspark.sql import functions as f

In [0]:
# Download BTS data from Jan 2019 to latest available month
# The script automatically detects the latest available month from BTS TranStats
# Already-downloaded months are skipped automatically for resume capability

from datetime import date
from dateutil.relativedelta import relativedelta

print("Starting BTS data download...")
print("Source: BTS TranStats On-Time Performance API")
print("Date range: Jan 2019 to latest available month (dynamic)")
print("Output: /Volumes/skyops/bronze/bts/staging/*.csv")
print("Note: Already-downloaded months will be skipped automatically")
print("=" * 60)

print("\nAttempting download with explicit start month: 2019-01")
print("End month: detected dynamically from BTS")
print("Expected runtime: 3-4 hours for initial full download (without a provisioned cluster)")
print("=" * 60 + "\n")

# Try explicit start month (recommended approach)
result = subprocess.run([
    sys.executable,
    "/Workspace/Users/hrpapasani@gmail.com/SkyOps/ingestion/bts_downloader.py",
    "--start-month", "2019-01",
    "--output-dir", "/Volumes/skyops/bronze/bts"
], capture_output=True, text=True)

# Check if the automatic detection failed
if result.returncode != 0 and "Latest Available Data" in result.stderr:
    print("⚠ Automatic month detection failed (BTS website may have changed)")
    print("Falling back to explicit month download...\n")
    
    # Generate month list from Jan 2019 to 6 months ago (conservative to avoid future months)
    start_date = date(2019, 1, 1)
    # Use a conservative end date: 6 months ago from today to avoid attempting future months
    end_date = date.today() - relativedelta(months=6)
    months_to_download = []
    current = start_date
    while current <= end_date:
        months_to_download.append(f"{current.year}-{current.month:02d}")
        current = current + relativedelta(months=1)
    
    print(f"Downloading {len(months_to_download)} months explicitly (up to 6 months ago)...")
    print("Note: BTS may have more recent data available - check manually if needed.")
    failed_months = []
    
    for month_str in months_to_download:
        month_result = subprocess.run([
            sys.executable,
            "/Workspace/Users/hrpapasani@gmail.com/SkyOps/ingestion/bts_downloader.py",
            "--month", month_str,
            "--output-dir", "/Volumes/skyops/bronze/bts"
        ], capture_output=True, text=True)
        
        # Show progress
        if "SKIP" in month_result.stdout or "DONE" in month_result.stdout:
            for line in month_result.stdout.split("\n"):
                if "SKIP" in line or "DONE" in line:
                    print(line)
                    break
        
        if month_result.returncode != 0:
            print(f"FAILED: {month_str}")
            failed_months.append(month_str)
    
    if failed_months:
        print(f"\n⚠ {len(failed_months)} months failed: {failed_months}")
        raise Exception("Download incomplete. Check failed months.")
else:
    # Success or different error
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise Exception(f"Download failed with return code {result.returncode}")

print("\n" + "=" * 60)
print("✓ Download complete! Staging CSVs ready for Delta table creation")
print("=" * 60)

In [0]:
dbutils.fs.ls("/Volumes/skyops/bronze/bts/staging")

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import to_date, year, month, col

print("="*70)
print("BRONZE LAYER: Creating Delta table from staging CSVs")
print("="*70)
print("Using explicit schema to prevent inference drift\n")

# Define explicit schema (matches normalized BTS data)
# FL_NUMBER is DoubleType to handle August 2024's decimal format (4934.0)
schema = StructType([
    StructField("FL_DATE", StringType(), True),           # yyyyMMdd format
    StructField("AIRLINE_CODE", StringType(), True),
    StructField("DOT_CODE", IntegerType(), True),
    StructField("FL_NUMBER", DoubleType(), True),         # ← CHANGED: DoubleType to handle "4934.0" format
    StructField("OriginAirportID", IntegerType(), True),
    StructField("DestAirportID", IntegerType(), True),
    StructField("ORIGIN_CITY", StringType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY", StringType(), True),
    StructField("CRS_DEP_TIME", StringType(), True),
    StructField("DEP_TIME", StringType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("WHEELS_OFF", StringType(), True),
    StructField("WHEELS_ON", StringType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", StringType(), True),
    StructField("ARR_TIME", StringType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("CANCELLATION_CODE", StringType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True),
    StructField("DELAY_DUE_CARRIER", DoubleType(), True),
    StructField("DELAY_DUE_WEATHER", DoubleType(), True),
    StructField("DELAY_DUE_NAS", DoubleType(), True),
    StructField("DELAY_DUE_SECURITY", DoubleType(), True),
    StructField("DELAY_DUE_LATE_AIRCRAFT", DoubleType(), True),
    StructField("Tail_Number", StringType(), True),
])

# Read all staging CSVs with explicit schema
print("Reading staging CSV files...")
df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("/Volumes/skyops/bronze/bts/staging/*.csv")

print(f"Loaded DataFrame with {len(df.columns)} columns")

# Cast FL_NUMBER from Double to Integer (handles August 2024's "4934.0" format)
print("Casting FL_NUMBER from Double to Integer...")
df = df.withColumn("FL_NUMBER", col("FL_NUMBER").cast(IntegerType()))

# Add YEAR and MONTH partition columns from FL_DATE (yyyyMMdd)
print("Adding YEAR and MONTH partition columns...")
df = df.withColumn(
    "FL_DATE_PARSED", 
    to_date(col("FL_DATE"), "yyyyMMdd")
)
df = df.withColumn("YEAR", year(col("FL_DATE_PARSED")))
df = df.withColumn("MONTH", month(col("FL_DATE_PARSED")))
df = df.drop("FL_DATE_PARSED")  # Drop temp column

# Write directly as managed Delta table (UC handles storage)
table_name = "skyops.bronze.bts_ontime"
print(f"\nWriting to Bronze Delta table: {table_name}")
print("Layer: BRONZE (raw data with minimal transformation)")
print("Partitioned by YEAR for query performance...")
print("This may take 10-15 minutes...\n")

df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("YEAR") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Bronze Delta table created successfully: {table_name}")
print(f"  Layer: BRONZE (raw ingestion)")
print(f"  Partitioned by: YEAR")
print(f"  Features: ACID, time travel, schema evolution")
print(f"\nVerifying row count...")
final_count = spark.table(table_name).count()
print(f"✓ Final row count: {final_count:,}")
print(f"\n✓✓✓ Bronze layer complete! Ready for Silver transformation.")
print("="*70)

In [0]:
df = spark.table("skyops.bronze.bts_ontime")

In [0]:
# %sql
# -- ANALYZE TABLE skyops.bronze.bts_ontime COMPUTE DELTA STATISTICS

In [0]:
# =============================================================================
# DATA INTEGRITY VALIDATION - Part 1: Table Metadata & Schema
# =============================================================================

print("\n" + "="*70)
print("VALIDATION PART 1: TABLE METADATA & STRUCTURE")
print("="*70 + "\n")

# 1. Confirm table exists and is accessible
try:
    table_info = spark.sql("DESCRIBE DETAIL skyops.bronze.bts_ontime").collect()[0]
    print("✓ Table exists: skyops.bronze.bts_ontime")
    print(f"  Format: {table_info['format']}")
    print(f"  Location: {table_info['location']}")
    print(f"  Created at: {table_info['createdAt']}")
    print(f"  Number of files: {table_info['numFiles']}")
    print(f"  Size on disk: {table_info['sizeInBytes'] / (1024**3):.2f} GB")
except Exception as e:
    print(f"✗ ERROR: Table not found or inaccessible: {e}")
    raise

print("\n" + "-"*70)

# 2. Verify schema structure
print("\n✓ Schema Validation:")
df_schema = spark.table("skyops.bronze.bts_ontime")
expected_columns = [
    "FL_DATE", "AIRLINE_CODE", "DOT_CODE", "FL_NUMBER",
    "OriginAirportID", "DestAirportID", "ORIGIN_CITY", "DEST", "DEST_CITY",
    "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY", "TAXI_OUT",
    "WHEELS_OFF", "WHEELS_ON", "TAXI_IN",
    "CRS_ARR_TIME", "ARR_TIME", "ARR_DELAY",
    "CANCELLED", "CANCELLATION_CODE", "DIVERTED",
    "CRS_ELAPSED_TIME", "ELAPSED_TIME", "AIR_TIME", "DISTANCE",
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
    "Tail_Number", "YEAR", "MONTH"
]

actual_columns = df_schema.columns
print(f"  Expected columns: {len(expected_columns)}")
print(f"  Actual columns: {len(actual_columns)}")

if set(expected_columns) == set(actual_columns):
    print("  ✓ All expected columns present")
else:
    missing = set(expected_columns) - set(actual_columns)
    extra = set(actual_columns) - set(expected_columns)
    if missing:
        print(f"  ✗ Missing columns: {missing}")
    if extra:
        print(f"  ✗ Extra columns: {extra}")

print("\n" + "-"*70)

In [0]:
# =============================================================================
# DATA INTEGRITY VALIDATION - Part 2: Row Count & Completeness
# =============================================================================

from pyspark.sql.functions import col

print("\n" + "="*70)
print("VALIDATION PART 2: ROW COUNT & DATA INTEGRITY")
print("="*70 + "\n")

df = spark.table("skyops.bronze.bts_ontime")

# 1. Count rows in staging CSVs vs Delta table
print("1. Comparing staging CSVs vs Delta table row counts...")
staging_path = "/Volumes/skyops/bronze/bts/staging/*.csv"
staging_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(staging_path)
staging_count = staging_df.count()
table_count = df.count()

print(f"   Staging CSVs:  {staging_count:>12,} rows")
print(f"   Delta Table:   {table_count:>12,} rows")
print(f"   Difference:    {abs(staging_count - table_count):>12,} rows")

if staging_count == table_count:
    print("   ✓ PERFECT MATCH: No data loss during ingestion!")
else:
    loss_pct = abs(staging_count - table_count) / staging_count * 100
    print(f"   ✗ WARNING: Row count mismatch ({loss_pct:.4f}% difference)")

print("\n" + "-"*70)

# 2. Check for completely empty rows
print("\n2. Checking for completely empty rows...")
empty_check = df.filter(
    (col("FL_DATE").isNull()) & 
    (col("AIRLINE_CODE").isNull()) & 
    (col("FL_NUMBER").isNull())
)
empty_count = empty_check.count()
if empty_count == 0:
    print(f"   ✓ No completely empty rows")
else:
    print(f"   ✗ WARNING: {empty_count:,} completely empty rows found")

print("\n" + "-"*70)

# 3. Verify date range coverage
print("\n3. Checking date range coverage...")
date_range = df.selectExpr(
    "MIN(FL_DATE) as min_date",
    "MAX(FL_DATE) as max_date",
    "COUNT(DISTINCT FL_DATE) as distinct_dates",
    "COUNT(DISTINCT YEAR) as distinct_years"
).collect()[0]

print(f"   Date range: {date_range['min_date']} to {date_range['max_date']}")
print(f"   Distinct dates: {date_range['distinct_dates']:,}")
print(f"   Distinct years: {date_range['distinct_years']}")
print(f"   Expected start: 20190101")

if date_range['min_date'] == '20190101':
    print("   ✓ Start date correct (Jan 1, 2019)")
else:
    print(f"   ⚠ Unexpected start date: {date_range['min_date']}")

print("\n" + "="*70)

In [0]:
# =============================================================================
# DATA INTEGRITY VALIDATION - Part 3: Data Quality Checks
# =============================================================================

from pyspark.sql.functions import col

print("\n" + "="*70)
print("VALIDATION PART 3: DATA QUALITY CHECKS")
print("="*70 + "\n")

df = spark.table("skyops.bronze.bts_ontime")

# 1. Verify partition coverage (years)
print("1. Checking partition coverage (YEAR)...")
year_dist = df.groupBy("YEAR").count().orderBy("YEAR").collect()
print(f"   Found {len(year_dist)} distinct years:")
for row in year_dist:
    print(f"     {row['YEAR']}: {row['count']:>12,} rows")

print("\n" + "-"*70)

# 2. Check key column null rates
print("\n2. Checking key column completeness...")
total_rows = df.count()
key_columns = ["FL_DATE", "AIRLINE_CODE", "FL_NUMBER", "ORIGIN_CITY", "DEST"]

for col_name in key_columns:
    null_count = df.filter(col(col_name).isNull()).count()
    null_pct = (null_count / total_rows) * 100
    status = "✓" if null_pct < 1 else "⚠"
    print(f"   {status} {col_name:20s}: {null_pct:6.2f}% null ({null_count:,} rows)")

print("\n" + "-"*70)

# 3. Check airlines and routes representation
print("\n3. Checking data distribution...")
airlines = df.select("AIRLINE_CODE").distinct().count()
routes = df.select("ORIGIN_CITY", "DEST_CITY").distinct().count()
airports = df.select("ORIGIN_CITY").distinct().count()

print(f"   Total distinct airlines: {airlines}")
print(f"   Total distinct routes: {routes:,}")
print(f"   Total distinct airports: {airports}")

print("\n" + "="*70)
print("\n✓✓✓ DATA QUALITY VALIDATION COMPLETE")
print("="*70)

# ✓ Bronze Layer Ingestion Complete

## Summary

This notebook successfully:

* **Downloaded BTS OnTime Performance data** from Jan 2019 to the latest available month using the `bts_downloader.py` script
* **Created Bronze Delta table** `skyops.bronze.bts_ontime` with explicit schema enforcement and YEAR partitioning
* **Validated data integrity** across three dimensions:
  * **Schema & Metadata**: Confirmed 34 columns (32 business + 2 partition), Delta format, table properties
  * **Row Count & Completeness**: Verified staging CSVs match Delta table, no empty rows, correct date range
  * **Data Quality**: Checked partition distribution, key column null rates, airline/route/airport coverage

## Bronze Table Details

* **Table**: `skyops.bronze.bts_ontime`
* **Layer**: Bronze (raw ingestion with minimal transformation)
* **Format**: Managed Delta Table (Unity Catalog)
* **Partitioning**: YEAR column for query performance
* **Schema**: 32 business columns + YEAR + MONTH
* **Date Range**: Jan 2019 to latest available month (dynamic)

## Medallion Architecture Progress

```
✓ RAW: BTS TranStats API
✓ BRONZE: skyops.bronze.bts_ontime (this notebook)
◻ SILVER: Data cleansing & quality (next step)
◻ GOLD: Business metrics (future)
```

## Next Steps

1. Create Silver layer notebook for data cleansing and enrichment
2. Parse FL_DATE to proper date/timestamp types
3. Add timezone conversions for flight times
4. Enrich with airport/airline reference data
5. Apply data quality rules and standardization

---

**Query the Bronze table**:
```sql
SELECT * FROM skyops.bronze.bts_ontime LIMIT 10
```